## Condemned and Dead-End Houses Data Cleaning

In [1]:
import pandas as pd

output_columns = [
    "_id",
    "parcel_id",
    "address",
    "zip_code",
    "owner",
    "property_type",
    "create_date",
    "latest_inspection_result",
    "latest_inspection_score",
    "record_number",
    "inspection_status",
    "latitude",
    "longitude",
    "neighborhood",
]

df = pd.read_csv("../data/raw.csv", dtype={"parcel_id": str})

df = (df
    .replace("", pd.NA)
    .dropna(how="all")
    .assign(
        create_date=lambda x: pd.to_datetime(x["create_date"], errors="coerce"),
        _id_num=lambda x: pd.to_numeric(x["_id"], errors="coerce"),
    )
)

# Keep the most recent record with a valid inspection score for each parcel.
df = (df[df["latest_inspection_score"].notna()]
    .sort_values(["parcel_id", "create_date", "_id_num"], ascending=[True, False, False], na_position="last")
    .drop_duplicates(subset="parcel_id", keep="first")
    .drop(columns="_id_num")
    .loc[:, output_columns]
    .reset_index(drop=True)
)

df.to_csv("../data/cleaned_data.csv", index=False)

df.head()

,_id,parcel_id,address,zip_code,owner,property_type,create_date,latest_inspection_result,latest_inspection_score,record_number,inspection_status,latitude,longitude,neighborhood
0,20442,0001H00046000000,"1ST AVE, Pittsburgh, PA 15222",15222,TROY DEVELOPMENT ASSOCIATES,Condemned/Dead End Property,2023-11-29,Fail,1.0,0001H00046000000,Active,40.438583,-80.003605,Central Business District
1,20593,0002F00330000000,"704 WATSON ST, Pittsburgh, PA 15282",15219,DUQUESNE UNIVERSITY OF THE HOLY GHOST,Condemned/Dead End Property,2025-03-27,Fail,1.0,0002F00330000000,Active,40.438175,-79.993168,Bluff
2,22427,0002G00051000000,"1334 5TH AVE, Pittsburgh, PA 15219",15219,1334 5TH AVENUE LLC,Condemned/Dead End Property,2024-07-19,Fail,3.0,0002G00051000000,Active,40.438386,-79.986986,Bluff
3,22600,0002G00056000000,"1325 5TH AVE, Pittsburgh, PA 15219",15219,URBAN REDEVELOPMENT AUTHORITY OFPITTSBURGH,Condemned/Dead End Property,2022-08-11,Fail,3.0,0002G00056000000,Active,40.438864,-79.987369,Crawford-Roberts
4,22052,0002G00057000000,"1319 5TH AVE, Pittsburgh, 15219",15219,URBAN REDEVELOPMENT AUTHORITY OFPITTSBURGH,Condemned/Dead End Property,2020-09-11,Fail,3.0,0002G00057000000,Active,40.438868,-79.987456,Crawford-Roberts


In [8]:
# Calculate the proportion of "No primary address specified"
no_address_count = (df['address'] == "No primary address specified").sum()
total_count = len(df)
percentage = (no_address_count / total_count) * 100

print(f"Total records: {total_count}")
print(f"Number of 'No primary address specified': {no_address_count}")
print(f"Proportion: {percentage:.2f}%")

Total records: 2665
Number of 'No primary address specified': 492
Proportion: 18.46%
